# Reconciling Campaign Runs onto the Calendar

This notebook demonstrates `solsys_code/campaign_reconciler.py` and
`solsys_code/management/commands/reconcile_campaign_runs.py` (Phase 29, "the reconciler"),
the single idempotent function/command pair that projects and refreshes `CalendarEvent`
rows for every `CampaignRun`, replacing the retired per-gap range-window backfill
command.

It demonstrates:

- Seeding two `Observatory` rows (a ground site with a real IANA timezone, and a satellite
  site) and a campaign `TargetList`
- Seeding three `CampaignRun` rows that exercise the reconciler's three dispatch branches:
  a classically-scheduled multi-night run, a queue-scheduled run, and a class-wide run
- Why `CampaignRun.source` must be set explicitly for the queue/classical split to render
  correctly against real (pre-v2.2) data
- A `--dry-run` sweep via `call_command`, showing the `would_create` counters and that no
  `CalendarEvent` rows are written
- A real sweep, then a printed loop over the resulting events showing the two coexisting
  key families: date-bearing `RUN:{pk}:{date}` rows for the classical run, and a single
  bare `RUN:{pk}` row each for the queue and class-wide runs
- A second real sweep reporting `created: 0, updated: 0` -- the idempotency claim
  demonstrated, not just asserted

This notebook lives in `pre_executed/` because it is **DB-dependent** (it seeds
`Observatory`/`TargetList` records and creates `CampaignRun`/`CalendarEvent` rows) and is
therefore **NOT** run during Sphinx/CI/ReadTheDocs builds, per `docs/notebooks/README.md`.

## Django setup

Standard boilerplate to make `src.fomo.settings` importable from this notebook's
location (`docs/notebooks/pre_executed/` -- three levels under the repo root, so
`parents[2]` gives the repo root) and to allow synchronous ORM calls inside
Jupyter's async event loop.

In [1]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
if not (repo_root_path / 'manage.py').exists():
    raise RuntimeError(f'No manage.py at {repo_root_path}; run Jupyter from docs/notebooks/pre_executed/')
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

django.setup()

# This notebook intentionally imports only the campaign-coordination models and the
# reconciler below -- never the ephemeris view/computation modules, which trigger a large
# one-time SPICE kernel download on first import.

print(f'Django ready: settings module={os.environ["DJANGO_SETTINGS_MODULE"]!r}, repo_root={repo_root!r}')

Django ready: settings module='src.fomo.settings', repo_root='/home/tlister/git/fomo_devel/.claude/worktrees/agent-aa382729e1d9fb9d6'


## Seed Observatory records and the campaign TargetList

The reconciler needs one resolvable ground `Observatory` (with a real IANA `timezone`, so
`sun_event()` can compute dip-corrected sunset/sunrise) and one satellite `Observatory`
(`observations_type=SATELLITE_OBSTYPE`, no fixed horizon -- the reconciler's container branch
skips the per-night sun math for these entirely). `update_or_create` makes this cell
idempotent -- safe to re-run against any dev DB.

The campaign container is a `tom_targets.models.TargetList`, found-or-created by name.

In [2]:
from tom_targets.models import TargetList

from solsys_code.solsys_code_observatory.models import Observatory

ground_site, _ = Observatory.objects.update_or_create(
    obscode='X29',
    defaults=dict(
        name='Reconciler Demo Ground Site',
        short_name='RDGS',
        lat=-29.2567,
        lon=-70.7300,
        altitude=2347,
        timezone='America/Santiago',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(f'Ground site:    obscode={ground_site.obscode!r}  timezone={ground_site.timezone!r}')

satellite_site, _ = Observatory.objects.update_or_create(
    obscode='X30',
    defaults=dict(
        name='Reconciler Demo Space Telescope',
        short_name='RDST',
        observations_type=Observatory.SATELLITE_OBSTYPE,
    ),
)
print(f'Satellite site: obscode={satellite_site.obscode!r}  observations_type=SATELLITE_OBSTYPE')

campaign, campaign_created = TargetList.objects.get_or_create(name='Reconciler Demo Campaign')
print(f'\nCampaign: {campaign.name!r} (pk={campaign.pk}) {"created" if campaign_created else "found"}')

Ground site:    obscode='X29'  timezone='America/Santiago'
Satellite site: obscode='X30'  observations_type=SATELLITE_OBSTYPE

Campaign: 'Reconciler Demo Campaign' (pk=1) created


## Seed three CampaignRun rows -- the three dispatch branches

`reconcile_run()` dispatches on the run's own state, in this order: a non-blank
`telescope_class` (class-wide/space) always wins first, then a satellite `site`, then
`source in QUEUE_SOURCES`, and only then the classical per-night branch. The three rows
below exercise all three outcomes:

1. **Classical** -- a non-queue `source`, a resolved ground site, and a 3-night window.
   Projects one `CalendarEvent` per observing night.
2. **Queue** -- `source` set explicitly to `CampaignRun.Source.LCO_QUEUE`. Projects a
   single bare whole-window container event.
3. **Class-wide** -- `telescope_class` set, no site at all. Also projects a single bare
   whole-window container event.

**Why `source` is set explicitly here, and what it means for real data:** the reconciler
branches purely on `CampaignRun.source` (never a text heuristic over
`telescope_instrument`/`site_raw`) -- see `campaign_reconciler.QUEUE_SOURCES`. The real
pre-v2.2 rows in the dev database still carry the model default, `source = legacy`, which
is a non-queue value. That means a genuine LCO/Gemini queue run imported before this
milestone will render through the *classical* per-night branch until a staff member sets
its `source` to the correct queue value through the Django admin -- only then does a real
sweep render the queue-versus-classical split correctly. See "How do I get every campaign
run onto the calendar?" in `docs/runbooks/telescope_runs_calendar.rst` for the operator-
facing version of this note.
**What happens if that correction happens *after* the run was already reconciled
once under its old classification:** `reconcile_run()` re-derives which calendar-event
family a run belongs to from its current `source`/`telescope_class`/`site` on every
call, so a later reconcile automatically detaches (never deletes) the old family's
events -- returning them to the attribution page's worklist for a staff member to
re-confirm or discard -- rather than leaving them on the calendar looking like a live
commitment forever. See `campaign_reconciler._detach_stale_family_events()` and "Can I
correct a run's source?" in `docs/runbooks/telescope_runs_calendar.rst` for the full
explanation.

In [3]:
from datetime import date

from solsys_code.models import CampaignRun

classical_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='RDGS/EFOSC2',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 3),
    defaults=dict(
        site=ground_site,
        site_raw='X29',
        observation_details='Classical multi-night photometric monitoring (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.CLASSICAL_FILE,
    ),
)
print(
    f'Classical run: pk={classical_run.pk}  source={classical_run.source!r}  '
    f'window={classical_run.window_start}..{classical_run.window_end}'
)

queue_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='RDGS 1m0-SciCam-Sinistro',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 10),
    defaults=dict(
        site=ground_site,
        site_raw='X29',
        observation_details='LCO queue allocation (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.LCO_QUEUE,
    ),
)
print(
    f'Queue run:     pk={queue_run.pk}  source={queue_run.source!r}  '
    f'window={queue_run.window_start}..{queue_run.window_end}'
)

class_wide_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='LCO 1m0 Network (demo)',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 30),
    defaults=dict(
        site=None,
        telescope_class=CampaignRun.TelescopeClass.ONE_M0,
        observation_details='Class-wide 1m0 network allocation (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.LEGACY,
    ),
)
print(
    f'Class-wide run: pk={class_wide_run.pk}  telescope_class={class_wide_run.telescope_class!r}  '
    f'site={class_wide_run.site!r}  window={class_wide_run.window_start}..{class_wide_run.window_end}'
)

Classical run: pk=1  source=CampaignRun.Source.CLASSICAL_FILE  window=2026-09-01..2026-09-03
Queue run:     pk=2  source=CampaignRun.Source.LCO_QUEUE  window=2026-09-01..2026-09-10
Class-wide run: pk=3  telescope_class=CampaignRun.TelescopeClass.ONE_M0  site=None  window=2026-09-01..2026-09-30


## Dry run first

`--dry-run` reports the `would_create`/`would_update`/`would_leave_unchanged` counters
without writing a single `CalendarEvent` row -- always run this before a real sweep.

In [4]:
import io

from django.core.management import call_command
from tom_calendar.models import CalendarEvent

events_before_dry_run = CalendarEvent.objects.count()

stdout_buf = io.StringIO()
stderr_buf = io.StringIO()
call_command('reconcile_campaign_runs', '--dry-run', stdout=stdout_buf, stderr=stderr_buf)

print('stdout:', stdout_buf.getvalue())
if stderr_buf.getvalue():
    print('stderr:', stderr_buf.getvalue())

events_after_dry_run = CalendarEvent.objects.count()
print(f'CalendarEvent.objects.count() before dry run: {events_before_dry_run}')
print(f'CalendarEvent.objects.count() after dry run:  {events_after_dry_run}')
assert events_after_dry_run == events_before_dry_run, 'A --dry-run sweep must never write a CalendarEvent row'

stdout: Done (dry run). runs: 3, would_create: 5, would_update: 0, would_leave_unchanged: 0, skipped: 0, failed: 0, blocked: 0

CalendarEvent.objects.count() before dry run: 0
CalendarEvent.objects.count() after dry run:  0


## The real sweep

Running the same sweep without `--dry-run` now writes the calendar events. The printed
loop below inspects only the three runs seeded above (via `campaign_reconciler.owned_events()`,
the same ownership-scoped query the reconciler itself uses), making the two coexisting key
families directly visible: the classical run gets one date-bearing `RUN:{pk}:{date}` event
per night, while the queue and class-wide runs each get a single bare `RUN:{pk}` container
event spanning their whole window.

Submitters write a run's Telescope / Instrument as free text using `/` or `+` (for example `'RDGS/EFOSC2'`). The reconciler splits that text on the first delimiter into the calendar event's two separate `telescope` and `instrument` fields -- exactly what the event-detail pop-up renders as its "Telescope" and "Instrument" boxes. A value with no delimiter at all, like the queue and class-wide runs' `telescope_instrument` below, goes wholly into `telescope` with `instrument` left blank.


In [5]:
from solsys_code.campaign_reconciler import owned_events

stdout_buf_real = io.StringIO()
stderr_buf_real = io.StringIO()
call_command('reconcile_campaign_runs', stdout=stdout_buf_real, stderr=stderr_buf_real)

print('stdout:', stdout_buf_real.getvalue())
if stderr_buf_real.getvalue():
    print('stderr:', stderr_buf_real.getvalue())

for label, run in [('Classical', classical_run), ('Queue', queue_run), ('Class-wide', class_wide_run)]:
    print(f'--- {label} run (pk={run.pk}) ---')
    for ev in owned_events(run).order_by('start_time'):
        print(f'  url={ev.url!r}')
        print(f'    title={ev.title!r}')
        print(f'    telescope={ev.telescope!r}  instrument={ev.instrument!r}')
        print(f'    start={ev.start_time.isoformat()}  end={ev.end_time.isoformat()}')
    print()

stdout: Done. runs: 3, created: 5, updated: 0, unchanged: 0, skipped: 0, failed: 0, blocked: 0

--- Classical run (pk=1) ---
  url='RUN:1:2026-09-01'
    title='Reconciler Demo Campaign: RDGS/EFOSC2 (window 2026-09-01..2026-09-03)'
    telescope='RDGS'  instrument='EFOSC2'
    start=2026-09-01T22:35:09+00:00  end=2026-09-02T10:49:50+00:00
  url='RUN:1:2026-09-02'
    title='Reconciler Demo Campaign: RDGS/EFOSC2 (window 2026-09-01..2026-09-03)'
    telescope='RDGS'  instrument='EFOSC2'
    start=2026-09-02T22:35:39+00:00  end=2026-09-03T10:48:41+00:00
  url='RUN:1:2026-09-03'
    title='Reconciler Demo Campaign: RDGS/EFOSC2 (window 2026-09-01..2026-09-03)'
    telescope='RDGS'  instrument='EFOSC2'
    start=2026-09-03T22:36:09+00:00  end=2026-09-04T10:47:31+00:00

--- Queue run (pk=2) ---
  url='RUN:2'
    title='Reconciler Demo Campaign: RDGS 1m0-SciCam-Sinistro (window 2026-09-01..2026-09-10)'
    telescope='RDGS 1m0-SciCam-Sinistro'  instrument=''
    start=2026-09-01T00:00:00+00:00 

## Idempotency: a second sweep changes nothing

Running `reconcile_campaign_runs` again against unchanged run state must report
`created: 0, updated: 0` -- demonstrated below rather than asserted, per RECON-01.

In [6]:
stdout_buf_second = io.StringIO()
stderr_buf_second = io.StringIO()
call_command('reconcile_campaign_runs', stdout=stdout_buf_second, stderr=stderr_buf_second)

print('Second sweep stdout:', stdout_buf_second.getvalue())
if stderr_buf_second.getvalue():
    print('Second sweep stderr:', stderr_buf_second.getvalue())

assert 'created: 0' in stdout_buf_second.getvalue()
assert 'updated: 0' in stdout_buf_second.getvalue()

Second sweep stdout: Done. runs: 3, created: 0, updated: 0, unchanged: 5, skipped: 0, failed: 0, blocked: 0



## Summary

`reconcile_campaign_runs` is for sweeps and backfills, not routine use: `approve()`,
`_resolve_site()`, `mark_cancelled` and `mark_weather_failure` in `campaign_views.py` each
call `campaign_reconciler.reconcile_run()` directly, so a single run's calendar events are
reconciled immediately the moment a staff member takes one of those actions on it -- no
command run is needed for that common case. The command demonstrated above exists for the
less common cases: reconciling every run at once (for example, after a bulk site repair),
or catching up a run whose state changed outside those four staff actions.

See `docs/runbooks/telescope_runs_calendar.rst`, "How do I get every campaign run onto the
calendar?", for the operator-facing version of everything demonstrated in this notebook.